# Benchmark dataloaders of rare SNPs (<0.1% MAF) of 1000G dataset

In [1]:
from __future__ import annotations

!pip install -q lamindb
!pip install -q annbatch 
!pip install -q git+https://github.com/laminlabs/arrayloader-benchmarks
!pip install -q zarrs
!pip install -q scDataset

In [2]:
import json

## Arrayloaders

In [3]:
%%capture cap --no-stderr
!python -m arrayloader_benchmarks.annbatch.benchmark_annbatch --batch_size=1024 --chunk_size=128 --preload_nchunks=32 --n_samples=100000 --store_path /lustre/boost_ai/users/selman.ozleyen/annbatch/rare_maf_0.001/rare_maf_0.001_500k_shuffled.zarr

In [4]:
arrayloaders_128 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_128

2335.1724342592515

In [5]:
%%capture cap --no-stderr
!python -m arrayloader_benchmarks.annbatch.benchmark_annbatch --batch_size=1024 --chunk_size=256 --preload_nchunks=32 --n_samples=100000 --store_path /lustre/boost_ai/users/selman.ozleyen/annbatch/rare_maf_0.001/rare_maf_0.001_500k_shuffled.zarr

In [6]:
arrayloaders_256 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_256

2349.276695169719

In [7]:
%%capture cap --no-stderr
!python -m arrayloader_benchmarks.annbatch.benchmark_annbatch --batch_size=1024 --chunk_size=512 --preload_nchunks=32 --n_samples=100000 --store_path /lustre/boost_ai/users/selman.ozleyen/annbatch/rare_maf_0.001/rare_maf_0.001_500k_shuffled.zarr

In [8]:
arrayloaders_512 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_512

2302.819468738933

## MappedCollection

In [9]:
%%capture cap --no-stderr
!python -m arrayloader_benchmarks.mapped_collection.benchmark_mapped_collection --num_workers=8 --batch_size=32 --store_path /lustre/boost_ai/users/selman.ozleyen/annbatch/rare_maf_0.001  --n_samples=100000

In [ ]:
mapped_collection = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
mapped_collection

56.82774715666317

## scDataset

In [11]:
%%capture cap --no-stderr
!python -m arrayloader_benchmarks.scDataset.benchmark_scDataset --num_workers=8 --fetch_factor=16 --batch_size=64 --store_path /lustre/boost_ai/users/selman.ozleyen/annbatch/rare_maf_0.001  --n_samples=100000

In [12]:
sc_dataset = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset

554.2321005561034

In [16]:
%%capture cap --no-stderr
!python -m arrayloader_benchmarks.scDataset.benchmark_scDataset --num_workers=8 --batch_size=1024 --block_size=256 --fetch_factor=4 --store_path /lustre/boost_ai/users/selman.ozleyen/annbatch/rare_maf_0.001  --n_samples=100000

In [17]:
sc_dataset_matched = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset_matched

830.8782075522981

## Plot results

In [18]:
import pandas as pd

results = pd.DataFrame(
    {
        "loader": [
            "annbatch (chunk_size=128)",
            "annbatch (chunk_size=256)",
            "annbatch (chunk_size=512)",
            "MappedCollection",
            "scDataset (recommended settings)",
            "scDataset (annbatch matched settings)",
        ],
        "samples/sec": [
            arrayloaders_128,
            arrayloaders_256,
            arrayloaders_512,
            mapped_collection,
            sc_dataset,
            sc_dataset_matched,
        ],
    }
)
results

,loader,samples/sec
0,annbatch (chunk_size=128),2335.172434
1,annbatch (chunk_size=256),2349.276695
2,annbatch (chunk_size=512),2302.819469
3,MappedCollection,56.827747
4,scDataset (recommended settings),554.232101
5,scDataset (annbatch matched settings),830.878208
